# 05 · Esperimenti su Colab

Fa girare su Google Colab la parte "colab" di `coda_tabella3.py`: i 4 modelli allenati su ASVspoof 2019 LA
con RawBoost algoritmo 3 (SSI), come il modello DF di XLSR-MamBo, e il loro test su ASVspoof 2021 DF intero.
Intanto il fisso fa la parte "fisso" (TFPARN con RawBoost algoritmo 5 e test su ASVspoof 5).

**Prima di partire**
- Runtime → Cambia tipo di runtime → **A100** (o L4). La T4 non va bene: non supporta bene il bfloat16 che usiamo.
- Su Google Drive servono circa **60 GB liberi**: 7 GB per 2019 LA, 35 GB per 2021 DF, il resto per i checkpoint.
- Gli archivi vanno su Drive una volta sola. L'audio invece si estrae ogni volta sul disco di Colab, che è molto più veloce di Drive da leggere.
- Checkpoint, log e punteggi vanno su Drive (`MyDrive/tesi/artifacts`), così non si perdono se Colab si disconnette.
  Se succede, basta rilanciare il notebook dall'inizio: riparte dall'ultima epoca finita.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/tesi"
os.makedirs(DRIVE + "/archivi", exist_ok=True)
os.environ["THESIS_DATA_ROOT"] = "/content/data"            # audio estratto (disco di Colab)
os.environ["THESIS_ARTIFACTS_ROOT"] = DRIVE + "/artifacts"   # checkpoint e punteggi (Drive)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Codice

Scarica il repo (il ramo con `coda_tabella3.py`) e installa le librerie. PyTorch su Colab c'è già.

In [ ]:
RAMO = "claude/project-thread-awrzvp"
if not os.path.exists("/content/SB2526-MasterThesis"):
    !git clone -b {RAMO} https://github.com/espositic/SB2526-MasterThesis /content/SB2526-MasterThesis
%cd /content/SB2526-MasterThesis
!git pull
!pip install -q transformers huggingface_hub soundfile imageio-ffmpeg

## 2. Dati

Gli archivi si scaricano su Drive (solo la prima volta, il DF richiede un po') e si estraggono in `/content/data`
(a ogni sessione, circa 15-20 minuti in tutto).

In [ ]:
from pathlib import Path
from tesi import dati, percorsi

ARCHIVI = Path(DRIVE) / "archivi"
file_2019 = {"LA.zip": "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip"}
file_2021 = {"DF-keys-full.tar.gz": "https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz"}
for parte in range(4):
    nome = f"ASVspoof2021_DF_eval_part0{parte}.tar.gz"
    file_2021[nome] = f"https://zenodo.org/records/4835108/files/{nome}"

for nome, url in file_2019.items():
    dati.estrai(dati.scarica(url, ARCHIVI / nome), percorsi.ASVSPOOF2019)
for nome, url in file_2021.items():
    dati.estrai(dati.scarica(url, ARCHIVI / nome), percorsi.ASVSPOOF2021)

In [ ]:
# liste: train e dev di 2019 LA, 2021 DF eval intero. I file mancanti devono essere 0.
for split in ("train", "dev"):
    dati.salva_lista(dati.lista_asvspoof2019(split), f"asvspoof2019_{split}")
dati.salva_lista(dati.lista_asvspoof2021("DF"), "asvspoof2021df_eval")
for nome in ("asvspoof2019_train", "asvspoof2019_dev", "asvspoof2021df_eval"):
    print(nome, "file mancanti:", dati.file_mancanti(dati.carica_lista(nome)))

## 3. Training e test

Lancia la coda. L'output va anche in `MyDrive/tesi/coda_colab.log`.
Se Colab si disconnette, rilancia tutte le celle: i passi già fatti vengono saltati.

In [ ]:
!python coda_tabella3.py colab 2>&1 | tee -a {DRIVE}/coda_colab.log

## 4. Risultati

Le cartelle `artifacts/esperimenti/<modello>_asvspoof2019_rawboost_ssi/seed1/` su Drive vanno copiate nella stessa
cartella del fisso (`artifacts\esperimenti\`): così il notebook 04 le mette nella tabella insieme alle altre.

In [ ]:
import json
for cartella in sorted((percorsi.ESPERIMENTI).glob("*_rawboost_ssi/seed1")):
    fatto = cartella / "completato.json"
    print(cartella.parent.name, json.loads(fatto.read_text()) if fatto.exists() else "in corso",
          [f.name for f in cartella.glob("punteggi_*.csv")])